# Glassware Canonical Product Analysis

Phase 3 workflow for `Laboratory Glassware`:

1. Pull labeled subset from `PROPOSED_L5` + `PRODUCTS_LCG`
2. Normalize naming/attributes
3. Build canonical identity key
4. Select representative records
5. Assign canonical product IDs
6. Export canonical and membership tables

This notebook emphasizes deterministic, reproducible analytics with optional LLM assistance only for naming/refinement.

In [4]:
import json
import re
from hashlib import sha1
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "product_classifier_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from product_classifier_utils import get_snowflake_session

In [5]:
# ------------------
# Config
# ------------------
DB = "SNOWFLAKE_LEARNING_DB"
SCHEMA = "SMCMAHON_PRODUCTS"

SOURCE_PRODUCTS_TABLE = f"{DB}.{SCHEMA}.PRODUCTS_LCG"
SOURCE_LABELS_TABLE = f"{DB}.{SCHEMA}.PROPOSED_L5"
TARGET_LABEL = "Laboratory Glassware"

# Output options
OUT_DIR = PROJECT_ROOT / "artifacts" / "analysis" / "glassware_canonical"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_TO_SNOWFLAKE = False
CANONICAL_TABLE = f"{DB}.{SCHEMA}.CANONICAL_GLASSWARE_PRODUCTS"
MEMBERSHIP_TABLE = f"{DB}.{SCHEMA}.CANONICAL_GLASSWARE_MEMBERS"

# Core-attribute parsing behavior
REQUIRE_CONTAINER_TYPE = True
MIN_GROUP_SIZE_FOR_REVIEW = 2

# Source column aliases (case-insensitive) -> standardized working columns
SOURCE_COLUMN_ALIASES = {
    "PRODUCT_ID": ["product_id", "PRODUCT_ID"],
    "PRODUCT_NAME": ["product_name", "PRODUCT_NAME"],
    "DESCRIPTION": ["description", "DESCRIPTION"],
    "PRICING_STATUS_C": ["pricing_status_c", "PRICING_STATUS_C"],
    "LIST_PRICE_C": ["list_price_c", "LIST_PRICE_C"],
    "SPECIFICATION_ASSIGNMENTS": [
        "specification_assignments_c",
        "SPECIFICATION_ASSIGNMENTS_C",
        "specification_assignments",
        "SPECIFICATION_ASSIGNMENTS",
    ],
}

In [6]:
# ------------------
# Extract glassware-labeled records
# ------------------
sf = get_snowflake_session()
sf.sql(f"USE DATABASE {DB}").collect()
sf.sql(f"USE SCHEMA {SCHEMA}").collect()

query = f"""
SELECT
    p.*,
    l.PROPOSED_CLUSTER,
    l.PROPOSED_LABEL
FROM {SOURCE_LABELS_TABLE} l
JOIN {SOURCE_PRODUCTS_TABLE} p
  ON p.PRODUCT_ID = l.PRODUCT_ID
WHERE l.PROPOSED_LABEL = '{TARGET_LABEL.replace("'", "''")}'
"""

df_raw = sf.sql(query).to_pandas()
print(f"Rows loaded: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns)}")

# Build lower-case lookup for flexible column mapping.
col_lookup = {c.lower(): c for c in df_raw.columns}

def resolve_col(candidates: list[str]) -> str | None:
    for c in candidates:
        if c.lower() in col_lookup:
            return col_lookup[c.lower()]
    return None

std_df = pd.DataFrame(index=df_raw.index)
for std_col, aliases in SOURCE_COLUMN_ALIASES.items():
    src_col = resolve_col(aliases)
    if src_col is not None:
        std_df[std_col] = df_raw[src_col]

# Always retain proposed outputs from label table if present.
for c in ["PROPOSED_CLUSTER", "PROPOSED_LABEL"]:
    src = resolve_col([c])
    if src is not None:
        std_df[c] = df_raw[src]

required_std = ["PRODUCT_ID", "PRODUCT_NAME", "DESCRIPTION", "SPECIFICATION_ASSIGNMENTS"]
missing_std = [c for c in required_std if c not in std_df.columns]
if missing_std:
    raise ValueError(f"Missing required standardized columns: {missing_std}")

df = std_df.copy()
print("Standardized columns:", sorted(df.columns))
df.head(3)

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://scienceexchange.okta.com/app/snowflake/exktzixqmxM7e7kdB697/sso/saml?SAMLRequest=lZJLc9owFIX%2Fikdd27J5owEyDpSUmZC4QLrITrUvIGxLRlfGTn59ZR6ddJHMdOeRz9F3dM8d3dV55pxAo1ByTALPJw7IWCVC7sbkZTN3B8RBw2XCMyVhTN4Ayd1khDzPChaWZi9XcCwBjWMvksiaH2NSaskUR4FM8hyQmZitw%2BUja3k%2B44igjcWRqyVBYVl7YwpGaVVVXtX2lN7Rlu%2F71B9Sq2ok38gHRPE1o9DKqFhlN0tt3%2FQJIqB%2Bp0FYhSVEV%2BO9kJcRfEX5fREh%2B7HZRG70vN4QJ7y9bqokljnoNeiTiOFl9XgJgDZBkcbdfqc78Ep0gaNxAw%2BlqrYZTyFWeVEae61nv%2BgWEpqpnbDDWszGpEhF8tQ99MLo8LB6PS52g3UYPbfCWX8u8mR5gsPU%2FPxuwoeeVquVionz61Ztq6l2gVjCQjaFGnvkt3qu33bb%2FsbvsGDIukOvF7ReiTOzhQrJzdl5S42xsEMCqOM9lzvwVGr4OSQvCvo3P4U6Ne%2BiPub1sg%2F9NLnvDfsUUdGmN3JZHXYOoif%2FPZAR%2FWi%2FruGTbWYxi1Qm4jdnrnTOzefFBV5wPhGJuz1LGeRcZGGSaEC0BWaZqqYauLHbbnQJhE4u1H%2F3ffIH&RelayState=ver%3A3-hint%3A9376132675904534-ETMsDgAAAZ08%2BE%2F8ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAECOUYk%2BkhveLb969xubKFqAAAACgjzx4iZKpo

,PRODUCT_ID,PRODUCT_NAME,DESCRIPTION,PRICING_STATUS_C,LIST_PRICE_C,SPECIFICATION_ASSIGNMENTS,PROPOSED_CLUSTER,PROPOSED_LABEL
0,01tPK00000HeAlHYAV,8oz (240ml) Natural HDPE Dairy Jug with 38-400...,Size: | Additional Shipping for Hazmat and o...,Requires Quote,NaN,"{""Size"":"""",""Additional Shipping for Hazmat an...",19,Laboratory Glassware
1,01tPK00000HeX9OYAV,"Bottle, Wide Mouth Round, Amber HDPE with Ambe...",Size: | Additional Shipping for Hazmat and o...,Requires Quote,NaN,"{""Size"":"""",""Additional Shipping for Hazmat an...",19,Laboratory Glassware
2,01tPK00000HeEn0YAF,55-Gallon Drum Spill Control Kit - Oil Only Ap...,Size: | Additional Shipping for Hazmat and o...,Requires Quote,NaN,"{""Size"":"""",""Additional Shipping for Hazmat an...",19,Laboratory Glassware


In [7]:
# ------------------
# Normalization helpers
# ------------------
CONTAINER_TYPES = [
    "beaker", "flask", "bottle", "vial", "tube", "jar", "cylinder", "pipette", "buret", "dish", "funnel"
]

# Subtypes capture shape/form inside a broad family.
SUBTYPE_PATTERNS = [
    "boston round",
    "french square",
    "erlenmeyer",
    "volumetric",
    "round bottom",
    "straight sided round",
    "wide mouth",
    "graduated",
    "desiccator",
    "precision point",
]

MATERIAL_KEYWORDS = {
    "borosilicate": "borosilicate_glass",
    "pyrex": "borosilicate_glass",
    "quartz": "quartz_glass",
    "soda lime": "soda_lime_glass",
    "amber": "amber_glass",
    "glass": "glass_unspecified",
}

VOL_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(ml|mL|l|L|ul|uL|µl|µL)")
DIM_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(mm|cm|in|\")")
PACK_RE = re.compile(r"(?:pack|pk|case|box|qty|quantity|size quantity)\s*[:x\- ]\s*(\d+)", re.IGNORECASE)

# Key normalization for inconsistent specification_assignments JSON keys.
SPEC_KEY_MAP = {
    "product size": "size_value",
    "size": "size_value",
    "capacity": "size_value",
    "volume": "size_value",
    "size quantity": "size_quantity",
    "qty": "size_quantity",
    "quantity": "size_quantity",
    "material": "material",
    "type": "container_type",
    "product type": "container_type",
}


def normalize_text(s: str) -> str:
    s = str(s or "")
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s


def parse_spec_json(value) -> dict:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    if isinstance(value, dict):
        return value
    txt = str(value).strip()
    if not txt:
        return {}
    try:
        parsed = json.loads(txt)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def normalize_spec_keys(spec: dict) -> dict:
    out = {}
    for k, v in spec.items():
        nk = normalize_text(k).replace("_", " ")
        nk = re.sub(r"\s+", " ", nk)
        std_k = SPEC_KEY_MAP.get(nk, nk)
        out[std_k] = v
    return out


def detect_container_type(text: str, spec_norm: dict | None = None) -> str | None:
    if spec_norm:
        st = normalize_text(spec_norm.get("container_type", ""))
        if st:
            for t in CONTAINER_TYPES:
                if t in st:
                    return t
    for t in CONTAINER_TYPES:
        if re.search(rf"\b{re.escape(t)}s?\b", text):
            return t
    return None


def detect_material(text: str, spec_norm: dict | None = None) -> str | None:
    if spec_norm:
        sm = normalize_text(spec_norm.get("material", ""))
        for k, v in MATERIAL_KEYWORDS.items():
            if k in sm:
                return v
    for k, v in MATERIAL_KEYWORDS.items():
        if k in text:
            return v
    return None


def detect_container_subtype(text: str, spec_norm: dict | None = None) -> str | None:
    low = normalize_text(text)
    if spec_norm:
        low = f"{low} {normalize_text(spec_norm.get('container_type', ''))}".strip()
    for pat in SUBTYPE_PATTERNS:
        if pat in low:
            return pat
    return None


def parse_primary_volume_ml(text: str, spec_norm: dict | None = None) -> float | None:
    spec_candidates = []
    if spec_norm:
        spec_candidates.append(str(spec_norm.get("size_value", "")))
    spec_candidates.append(text)

    for cand in spec_candidates:
        m = VOL_RE.search(str(cand))
        if not m:
            continue
        val = float(m.group(1))
        unit = m.group(2).lower()
        if unit == "l":
            return val * 1000.0
        if unit in {"ul", "µl"}:
            return val / 1000.0
        return val
    return None


def parse_primary_dimension_mm(text: str) -> float | None:
    m = DIM_RE.search(text)
    if not m:
        return None
    val = float(m.group(1))
    unit = m.group(2).lower()
    if unit == "cm":
        return val * 10.0
    if unit in {'in', '"'}:
        return val * 25.4
    return val


def parse_pack_count(text: str, spec_norm: dict | None = None) -> int | None:
    if spec_norm:
        sq = normalize_text(spec_norm.get("size_quantity", ""))
        m2 = re.search(r"(\d+)", sq)
        if m2:
            return int(m2.group(1))
    m = PACK_RE.search(text)
    if m:
        return int(m.group(1))
    return None

In [8]:
# ------------------
# Feature extraction for canonicalization
# ------------------
name_text = df.get("PRODUCT_NAME", "").fillna("").astype(str)
desc_text = df.get("DESCRIPTION", "").fillna("").astype(str)
combined_text = (name_text + " " + desc_text).map(normalize_text)

df["NORM_TEXT"] = combined_text

df["SPEC_PARSED"] = df["SPECIFICATION_ASSIGNMENTS"].map(parse_spec_json)
df["SPEC_NORM"] = df["SPEC_PARSED"].map(normalize_spec_keys)
df["SPEC_SIZE_VALUE"] = df["SPEC_NORM"].map(lambda d: d.get("size_value") if isinstance(d, dict) else None)
df["SPEC_SIZE_QUANTITY"] = df["SPEC_NORM"].map(lambda d: d.get("size_quantity") if isinstance(d, dict) else None)

df["CONTAINER_FAMILY"] = [detect_container_type(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["CONTAINER_SUBTYPE"] = [detect_container_subtype(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
# Backward-compatible alias used in downstream cells.
df["CONTAINER_TYPE"] = df["CONTAINER_FAMILY"]

df["MATERIAL_STD"] = [detect_material(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["VOLUME_ML"] = [parse_primary_volume_ml(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]
df["DIMENSION_MM"] = df["NORM_TEXT"].map(parse_primary_dimension_mm)
df["PACK_COUNT"] = [parse_pack_count(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])]

# Basic completeness score favors records useful as canonical exemplars.
score_cols = [
    "CONTAINER_FAMILY",
    "CONTAINER_SUBTYPE",
    "MATERIAL_STD",
    "VOLUME_ML",
    "DIMENSION_MM",
    "PACK_COUNT",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "PRODUCT_NAME",
    "DESCRIPTION",
]
df["COMPLETENESS_SCORE"] = sum(df[c].notna().astype(int) for c in score_cols if c in df.columns)

df[[
    "PRODUCT_ID",
    "CONTAINER_TYPE",
    "MATERIAL_STD",
    "VOLUME_ML",
    "DIMENSION_MM",
    "PACK_COUNT",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "COMPLETENESS_SCORE",
]].head(10)

,PRODUCT_ID,CONTAINER_TYPE,MATERIAL_STD,VOLUME_ML,DIMENSION_MM,PACK_COUNT,SPEC_SIZE_VALUE,SPEC_SIZE_QUANTITY,COMPLETENESS_SCORE
0,01tPK00000HeAlHYAV,None,None,240.0,NaN,NaN,,None,4
1,01tPK00000HeX9OYAV,bottle,amber_glass,30.0,NaN,NaN,,None,6
2,01tPK00000HeEn0YAF,None,None,NaN,NaN,NaN,,None,3
3,01tPK00000HenPHYAZ,flask,None,100.0,4.0,NaN,,None,6
4,01tPK00000HeAmsYAF,bottle,None,480.0,NaN,NaN,,None,5
5,01tPK00000HfaAQYAZ,flask,glass_unspecified,5000.0,NaN,NaN,,None,6
6,01tPK00000IawFAYAZ,flask,borosilicate_glass,10.0,115.0,NaN,None,None,6
7,01tPK00000HeDRcYAN,jar,None,NaN,NaN,NaN,,None,4
8,01tPK00000HeIyhYAF,flask,None,125.0,NaN,NaN,,None,5
9,01tPK00000HeFNGYA3,None,amber_glass,2500.0,NaN,NaN,,None,5


In [9]:
# ------------------
# Build canonical identity key + IDs
# ------------------

def canonical_key(row: pd.Series) -> str:
    family = row.get("CONTAINER_FAMILY") or "unknown"
    subtype = row.get("CONTAINER_SUBTYPE") or "unknown"
    mat = row.get("MATERIAL_STD") or "unknown"

    vol = row.get("VOLUME_ML")
    vol_bucket = "na" if pd.isna(vol) else str(int(round(float(vol))))

    dim = row.get("DIMENSION_MM")
    dim_bucket = "na" if pd.isna(dim) else str(int(round(float(dim))))

    # Use normalized size text as backup discriminator when numeric parsing is unavailable.
    size_txt = normalize_text(row.get("SPEC_SIZE_VALUE", ""))
    size_bucket = size_txt[:40] if size_txt else "na"

    # pack count is variant, not core identity.
    return "|".join([family, subtype, mat, vol_bucket, dim_bucket, size_bucket])


df["CANONICAL_KEY"] = df.apply(canonical_key, axis=1)

if REQUIRE_CONTAINER_TYPE:
    unknown_mask = df["CONTAINER_TYPE"].isna()
    print(f"Rows missing container type: {int(unknown_mask.sum()):,}")

# Stable canonical product ID from canonical key
unique_keys = sorted(df["CANONICAL_KEY"].dropna().unique())
key_to_id = {k: f"CGW_{sha1(k.encode('utf-8')).hexdigest()[:10].upper()}" for k in unique_keys}
df["CANONICAL_PRODUCT_ID"] = df["CANONICAL_KEY"].map(key_to_id)

print(f"Unique canonical groups: {df['CANONICAL_PRODUCT_ID'].nunique():,}")
df[["PRODUCT_ID", "CANONICAL_KEY", "CANONICAL_PRODUCT_ID"]].head(10)

Rows missing container type: 3,105
Unique canonical groups: 1,683


,PRODUCT_ID,CANONICAL_KEY,CANONICAL_PRODUCT_ID
0,01tPK00000HeAlHYAV,unknown|unknown|240|na|na,CGW_377EB322E9
1,01tPK00000HeX9OYAV,bottle|amber_glass|30|na|na,CGW_531CD32FBB
2,01tPK00000HeEn0YAF,unknown|unknown|na|na|na,CGW_AB863DE3CB
3,01tPK00000HenPHYAZ,flask|unknown|100|4|na,CGW_94EC83BAD9
4,01tPK00000HeAmsYAF,bottle|unknown|480|na|na,CGW_756C2E187A
5,01tPK00000HfaAQYAZ,flask|glass_unspecified|5000|na|na,CGW_285EA6E7D2
6,01tPK00000IawFAYAZ,flask|borosilicate_glass|10|115|na,CGW_37BE17BCF1
7,01tPK00000HeDRcYAN,jar|unknown|na|na|na,CGW_9B800FFB7B
8,01tPK00000HeIyhYAF,flask|unknown|125|na|na,CGW_02087C6314
9,01tPK00000HeFNGYA3,unknown|amber_glass|2500|na|na,CGW_6420AB1A38


In [10]:
# ------------------
# Select representative record per canonical group
# ------------------

sort_cols = ["CANONICAL_PRODUCT_ID", "COMPLETENESS_SCORE"]
rep_idx = (
    df.sort_values(sort_cols, ascending=[True, False])
    .groupby("CANONICAL_PRODUCT_ID", as_index=False)
    .head(1)
    .index
)

canonical_df = df.loc[rep_idx].copy()
canonical_df = canonical_df.rename(columns={
    "PRODUCT_ID": "REPRESENTATIVE_PRODUCT_ID",
    "PRODUCT_NAME": "CANONICAL_PRODUCT_NAME",
    "DESCRIPTION": "CANONICAL_DESCRIPTION",
})

# Group size and basic confidence signals
group_sizes = df.groupby("CANONICAL_PRODUCT_ID").size().rename("GROUP_SIZE")
canonical_df = canonical_df.merge(group_sizes, on="CANONICAL_PRODUCT_ID", how="left")
canonical_df["NEEDS_REVIEW"] = canonical_df["GROUP_SIZE"] < MIN_GROUP_SIZE_FOR_REVIEW

canonical_cols = [
    "CANONICAL_PRODUCT_ID",
    "CANONICAL_KEY",
    "REPRESENTATIVE_PRODUCT_ID",
    "CANONICAL_PRODUCT_NAME",
    "CANONICAL_DESCRIPTION",
    "CONTAINER_FAMILY",
    "CONTAINER_SUBTYPE",
    "MATERIAL_STD",
    "VOLUME_ML",
    "DIMENSION_MM",
    "SPEC_SIZE_VALUE",
    "SPEC_SIZE_QUANTITY",
    "GROUP_SIZE",
    "NEEDS_REVIEW",
]
canonical_cols = [c for c in canonical_cols if c in canonical_df.columns]
canonical_df = canonical_df[canonical_cols].sort_values(["GROUP_SIZE", "CANONICAL_PRODUCT_ID"], ascending=[False, True])

membership_df = df[[
    "PRODUCT_ID",
    "CANONICAL_PRODUCT_ID",
    "CANONICAL_KEY",
    "PROPOSED_CLUSTER",
    "PROPOSED_LABEL",
    "PACK_COUNT",
    "COMPLETENESS_SCORE",
]].copy()

print("Canonical table rows:", len(canonical_df))
print("Membership table rows:", len(membership_df))
canonical_df.head(10)

Canonical table rows: 1683
Membership table rows: 14815


,CANONICAL_PRODUCT_ID,CANONICAL_KEY,REPRESENTATIVE_PRODUCT_ID,CANONICAL_PRODUCT_NAME,CANONICAL_DESCRIPTION,CONTAINER_TYPE,MATERIAL_STD,VOLUME_ML,DIMENSION_MM,SPEC_SIZE_VALUE,SPEC_SIZE_QUANTITY,GROUP_SIZE,NEEDS_REVIEW
424,CGW_431ED0CC6E,bottle|unknown|na|na|na,01tPK00000HeQPcYAN,United 16OZ SPRAY BOTTLE WITH PUMP PK/8,Size: | Additional Shipping for Hazmat and o...,bottle,None,NaN,NaN,,None,693,False
25,CGW_04484FDB47,bottle|unknown|500|na|na,01tPK00000He8QBYAZ,BOTTLE 500ML. PK.4 (C003B-028182),Size: | Additional Shipping for Hazmat and o...,bottle,None,500.0,NaN,,None,424,False
1125,CGW_AB863DE3CB,unknown|unknown|na|na|na,01tPK00000HdlunYAB,"Poplypropylene Linerless Caps, Bulk Pack, 13-415",Size: | Additional Shipping for Hazmat and o...,None,None,NaN,NaN,,None,411,False
722,CGW_6C196C754A,flask|glass_unspecified|500|na|na,01tPK00000HerlvYAB,Glassware assembly for one sample position; 50...,Size: | Additional Shipping for Hazmat and o...,flask,glass_unspecified,500.0,NaN,,None,312,False
1341,CGW_CC5105DF0D,bottle|unknown|1000|na|na,01tPK00000IPXahYAH,GLOBE SCIENTIFIC DIAMOND REALSEAL BOTTLES (C00...,Product Size: | Additional Shipping for Hazma...,bottle,None,1000.0,NaN,,None,295,False
739,CGW_6DFFBC91C0,flask|glass_unspecified|1000|na|na,01tPK00000HfZ34YAF,"Flask, Shake,1L,Deep Baffles,Delong Neck",Size: | Additional Shipping for Hazmat and o...,flask,glass_unspecified,1000.0,NaN,,None,270,False
379,CGW_3A50DD86B8,flask|glass_unspecified|250|na|na,01tPK00000HerluYAB,Glassware assembly for one sample position; 25...,Size: | Additional Shipping for Hazmat and o...,flask,glass_unspecified,250.0,NaN,,None,262,False
371,CGW_3988F05B7D,bottle|unknown|250|na|na,01tPK00000IPXalYAH,GLOBE SCIENTIFIC DIAMOND REALSEAL BOTTLES (C00...,Product Size: | Additional Shipping for Hazma...,bottle,None,250.0,NaN,,None,261,False
513,CGW_5142EDE27E,flask|unknown|500|na|na,01tPK00000HeQhuYAF,"500ml Volumetric Flask w/Plastic cap, pk/12",Size: | Additional Shipping for Hazmat and o...,flask,None,500.0,NaN,,None,253,False
1338,CGW_CBFBBE2E25,flask|unknown|250|na|na,01tPK00000HeQbaYAF,250ml WM Erlenmeyer Flask Pk/12,Size: | Additional Shipping for Hazmat and o...,flask,None,250.0,NaN,,None,240,False


## Unknown Container-Type Review (run before save/persist)

Use these cells to inspect `UNKNOWN` container types and add targeted synonym rules.

Suggested loop:
1. Run unknown profiling cells
2. Add/adjust `CONTAINER_TYPE_SYNONYMS`
3. Rerun the reclassification cell
4. Rerun canonical ID + representative selection cells
5. Then run save/persist cells

In [24]:
# Profile unknown container-type records
unk = df[df["CONTAINER_TYPE"].isna()].copy()
print(f"Unknown rows: {len(unk):,} ({len(unk)/max(len(df),1):.1%})")

if len(unk) > 0:
    unk["PRODUCT_NAME_NORM"] = unk["PRODUCT_NAME"].fillna("").str.lower().str.strip()
    display(
        unk["PRODUCT_NAME_NORM"]
        .value_counts()
        .head(50)
        .rename_axis("product_name")
        .reset_index(name="count")
    )

    # Token frequency from unknown records
    text = (unk["PRODUCT_NAME"].fillna("") + " " + unk["DESCRIPTION"].fillna("")).str.lower()
    tokens = text.str.replace(r"[^a-z0-9 ]", " ", regex=True).str.split()

    from collections import Counter

    ctr = Counter()
    for row in tokens:
        ctr.update([t for t in row if len(t) >= 3])

    top_tokens_df = pd.DataFrame(ctr.most_common(100), columns=["token", "count"])
    display(top_tokens_df.head(40))
else:
    print("No unknowns to profile.")

Unknown rows: 1,773 (12.0%)


,product_name,count
0,leak-resistant design with double-seal closure...,2
1,55-gallon drum spill control kit - oil only ap...,1
2,"tank, 100 gal/378 l, 711 mm id x 1067 mm deep",1
3,allwik 55-gallon drum spill control kit refil...,1
4,"20l shaking water bath, to 99 c; 0.02 c stabil...",1
5,"kugelrohr distilling bulb, capacity: 15ml, joi...",1
6,250ml conical bottom,1
7,"glass mortar, 4 oz",1
8,refill-model pwb-2,1
9,kinsman weighted & non-weighted insulated bowl...,1


,token,count
0,for,1909
1,and,1872
2,size,1812
3,product,1787
4,manufacturer,1772
5,name,1771
6,number,1771
7,cenmed,1767
8,additional,1764
9,shipping,1764


In [23]:
# Optional manual synonyms for unknown classification.
# Add high-frequency patterns discovered above.
CONTAINER_TYPE_SYNONYMS = {
    # pattern : canonical_container_type
    "erlenmeyer": "flask",
    "round bottom": "flask",
    "volumetric flask": "flask",
    "volflask": "flask",
    "media bottle": "bottle",
    "reagent bottle": "bottle",
    "cryovial": "vial",
    "centrifuge tube": "tube",
    "petri": "dish",
    "boston round": "bottle",
    "packer": "bottle",
    "french square": "bottle",
    "straight sided round": "bottle",
    "wide mouth round": "bottle",
    "medium round": "bottle",
    "handled round jug": "bottle",
    "jug": "bottle", 
    "desiccator": "jar",
     "standard wide mouth": "bottle",
    "precision point": "bottle",
    "aerosol": "bottle"  # or "spray_bottle" if you want finer subtype

}


def detect_container_type_with_synonyms(text: str, spec_norm: dict | None = None) -> str | None:
    # 1) existing deterministic detector
    base = detect_container_type(text, spec_norm)
    if base is not None:
        return base

    # 2) synonym fallback
    low = normalize_text(text)
    for pattern, mapped in CONTAINER_TYPE_SYNONYMS.items():
        if pattern in low:
            return mapped
    return None


# Reclassify using synonym-extended detector
before_unknown = int(df["CONTAINER_TYPE"].isna().sum())
df["CONTAINER_TYPE"] = [
    detect_container_type_with_synonyms(t, s) for t, s in zip(df["NORM_TEXT"], df["SPEC_NORM"])
]
after_unknown = int(df["CONTAINER_TYPE"].isna().sum())

print(f"Unknown before: {before_unknown:,}")
print(f"Unknown after : {after_unknown:,}")
print(f"Recovered      : {before_unknown - after_unknown:,}")

display(
    df["CONTAINER_TYPE"].fillna("UNKNOWN").value_counts().rename_axis("container_type").reset_index(name="count")
)

Unknown before: 1,872
Unknown after : 1,773
Recovered      : 99


,container_type,count
0,bottle,5364
1,flask,5332
2,UNKNOWN,1773
3,beaker,904
4,jar,553
5,cylinder,326
6,vial,294
7,funnel,123
8,dish,69
9,tube,60


After adjusting `CONTAINER_TYPE_SYNONYMS`, rerun the canonical grouping cells (`Build canonical identity key + IDs` and `Select representative record`) before saving outputs.

In [21]:
# Unique container_type values + counts
container_summary = (
    df["CONTAINER_TYPE"]
    .fillna("UNKNOWN")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("CONTAINER_TYPE")
    .reset_index(name="COUNT")
)

display(container_summary)

,CONTAINER_TYPE,COUNT
0,flask,5332
1,bottle,5265
2,UNKNOWN,1903
3,beaker,904
4,jar,522
5,cylinder,326
6,vial,294
7,funnel,123
8,dish,69
9,tube,60


In [11]:
# ------------------
# Save outputs locally
# ------------------
canonical_csv = OUT_DIR / "canonical_glassware_products.csv"
membership_csv = OUT_DIR / "canonical_glassware_membership.csv"
summary_json = OUT_DIR / "canonical_glassware_summary.json"

canonical_df.to_csv(canonical_csv, index=False)
membership_df.to_csv(membership_csv, index=False)

summary = {
    "source_products_table": SOURCE_PRODUCTS_TABLE,
    "source_labels_table": SOURCE_LABELS_TABLE,
    "target_label": TARGET_LABEL,
    "rows_input": int(len(df)),
    "rows_canonical": int(len(canonical_df)),
    "rows_membership": int(len(membership_df)),
    "groups_needing_review": int(canonical_df["NEEDS_REVIEW"].sum()) if "NEEDS_REVIEW" in canonical_df.columns else None,
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:")
print("-", canonical_csv)
print("-", membership_csv)
print("-", summary_json)
summary

Saved:
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_products.csv
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_membership.csv
- /Users/stephanie.mcmahon/smcmahon_repo/auto_classification/artifacts/analysis/glassware_canonical/canonical_glassware_summary.json


{'source_products_table': 'SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PRODUCTS_LCG',
 'source_labels_table': 'SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.PROPOSED_L5',
 'target_label': 'Laboratory Glassware',
 'rows_input': 14815,
 'rows_canonical': 1683,
 'rows_membership': 14815,
 'groups_needing_review': 783}

In [12]:
# ------------------
# Optional write-back to Snowflake
# ------------------
if WRITE_TO_SNOWFLAKE:
    sf.sql(f"USE DATABASE {DB}").collect()
    sf.sql(f"USE SCHEMA {SCHEMA}").collect()

    sf_can = sf.create_dataframe(canonical_df)
    sf_mem = sf.create_dataframe(membership_df)

    sf_can.write.mode("overwrite").save_as_table(CANONICAL_TABLE)
    sf_mem.write.mode("overwrite").save_as_table(MEMBERSHIP_TABLE)

    print(f"Wrote: {CANONICAL_TABLE}")
    print(f"Wrote: {MEMBERSHIP_TABLE}")
else:
    print("Set WRITE_TO_SNOWFLAKE=True to publish canonical tables.")

Set WRITE_TO_SNOWFLAKE=True to publish canonical tables.
